# Sentiment Analysis - YouTube Comments (Indonesian)
**Nama**: Asep Haryana Saputra

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout, SpatialDropout1D, BatchNormalization
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils import resample
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import make_pipeline
from tqdm.auto import tqdm

print(f"TensorFlow Version: {tf.__version__}")

2026-02-10 20:21:43.725566: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-10 20:21:43.775208: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-02-10 20:21:44.876294: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow Version: 2.20.0


## 2. Data Loading & Preprocessing
**Target Classes**:
- 0: **Negative**
- 1: **Positive**
- 2: **Spam**

In [2]:
# Nama file dataset
filename = 'dataset_youtube.csv'

if not os.path.exists(filename):
    raise FileNotFoundError(f"File {filename} not found! Please run the scraping script first.")

df = pd.read_csv(filename)
print(f"Loaded {len(df)} rows from {filename}")

if 'label' in df.columns and df['label'].isin([0,1,2]).all():
    print("Using existing labels from dataset.")
    df_clean = df.copy()
else:
    print("Labeling data using heuristic (Auto-Labeling)...")
    # Auto-Labeling Function (Heuristic)
    def auto_label(text):
        text = str(text).lower()
        
        # SPAM Keywords
        spam_keywords = [
            'cek ig', 'promo', 'jual', 'obat', 'peninggi', 'pelangsing', 'subscribe', 'sub4sub', 
            'mampir', 'channel', 'link bio', 'giveaway', 'hadiah', 'gratis', 'follow', 'visit',
            'klik', 'top up', 'jasa', 'bisnis', 'wa', '08', 'judol', 'slot', 'gacor'
        ]
        if any(k in text for k in spam_keywords):
            return 2 # Spam
            
        # POSITIVE Keywords
        pos_keywords = [
            'bagus', 'keren', 'mantap', 'semangat', 'good', 'nice', 'love', 'thanks', 'terima kasih', 
            'bermanfaat', 'membantu', 'suka', 'cool', 'best', 'hebat', 'lanjutkan', 'panutan', 'idol',
            'legend', 'terhibur', 'lucu', 'ngakak', 'kocak', 'anjay', 'anjing' if 'keren' in text else '@@@', 
            'gokil', 'pecah', 'trending', 'karyamu', 'support', 'wow', 'gas', 'menyala', 'respect', 'salut',
            'kualitas', 'setuju', 'bantu'
        ]
        if any(k in text for k in pos_keywords):
            return 1 # Positive
            
        # NEGATIVE Keywords
        neg_keywords = [
            'jelek', 'buruk', 'sampah', 'gagal', 'benci', 'hate', 'bad', 'stupid', 'idiot', 'bodoh', 
            'gajelas', 'malas', 'susah', 'ribet', 'unsub', 'cringe', 'alay', 'lebay', 'norak', 
            'garing', 'toxic', 'kontol', 'memek', 'anjing', 'bangsat', 'tolol', 'goblok', 'hujat', 'drama',
            'settingan', 'sampah', 'kecewa', 'merugikan', 'penipu', 'bohong', 'palsu', 'hoax', 'fitnah'
        ]
        if any(k in text for k in neg_keywords):
            return 0 # Negative
            
        return -1 # Unlabeled

    df['label'] = df['text'].apply(auto_label)
    df_clean = df[df['label'] != -1].copy()

print(f"Labeled Data Before Balancing: {len(df_clean)} rows")
print(df_clean['label'].value_counts())

# --- BALANCING DATA (Upsampling) ---
# Separate majority and minority classes
df_pos = df_clean[df_clean.label==1]
df_neg = df_clean[df_clean.label==0]
df_spam = df_clean[df_clean.label==2]

# Upsample minority classes
n_samples = max(len(df_pos), len(df_neg), len(df_spam)) # Match majority class

df_pos_upsampled = resample(df_pos, replace=True, n_samples=n_samples, random_state=42)
df_neg_upsampled = resample(df_neg, replace=True, n_samples=n_samples, random_state=42)
df_spam_upsampled = resample(df_spam, replace=True, n_samples=n_samples, random_state=42)

# Combine majority class with upsampled minority classes
df_balanced = pd.concat([df_pos_upsampled, df_neg_upsampled, df_spam_upsampled])
df_clean = df_balanced # Replace working dataframe

print("\nAfter Balancing:")
print(df_clean['label'].value_counts())

df_clean.head()

Loaded 2333 rows from dataset_youtube.csv
Labeling data using heuristic (Auto-Labeling)...
Labeled Data Before Balancing: 1344 rows
label
1    1128
2     182
0      34
Name: count, dtype: int64

After Balancing:
label
1    1128
0    1128
2    1128
Name: count, dtype: int64


,cid,text,time,author,votes,photo,heart,label
2330,Ugw6jYgZJyApgmmedLV4AaABAg,Bikin ngakak anjay,5 tahun yang lalu,@bagusaditian2395,0,https://yt3.ggpht.com/ytc/AIdro_k42k38f7HaX6eV...,False,1
1786,UgySlgvosoHERTEugRp4AaABAg,"Anjay ,anjay ,anjay",5 tahun yang lalu,@muhamadtaufikpermana7857,0,https://yt3.ggpht.com/ytc/AIdro_kOxrkfAv-7MNF5...,False,1
2262,UgxVSUwSIoc7Eoj8PrN4AaABAg,ANJAY RIBET,5 tahun yang lalu,@Pajarmson.,0,https://yt3.ggpht.com/RiKSpZh3XLNRBPIG4_9nKFcW...,False,1
2159,Ugxd7nEPl7Xh288f9Dh4AaABAg,HBS INI TIAP DIKATAIN ANJAY BANYAK YG BAPERAN ...,5 tahun yang lalu,@joyashop1798,0,https://yt3.ggpht.com/ytc/AIdro_mkyUnk3Mb1aBHT...,False,1
267,UgwPwqqKva-HTdnXYTZ4AaABAg.9D50eRozUuj9D5RyxfNxHs,Anjay,5 tahun yang lalu,@cilalawcila,0,https://yt3.ggpht.com/Nxt1eaSmLPLlzODSddWNon-6...,False,1


In [3]:
# Text Cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text) # Remove special chars
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

df_clean['cleaned_text'] = df_clean['text'].apply(clean_text)
df_clean = df_clean[df_clean['cleaned_text'].str.len() > 3] # Min length filter

## 3. Modeling Schemes Comparison
We will test 3 different schemes:
1. **Traditional ML (SVM + TF-IDF)** - Target > 85%
2. **Alternative ML (Random Forest + CountVectorizer)** - Target > 85%
3. **Deep Learning (Bi-LSTM + Optimization)** - Target > 92%

### Scheme 1: SVM + TF-IDF (80/20 Tuning)

In [4]:
# Split Data
X_train_ml, X_test_ml, y_train_ml, y_test_ml = train_test_split(
    df_clean['cleaned_text'], df_clean['label'], test_size=0.2, random_state=42, stratify=df_clean['label']
)

# Pipeline: TF-IDF -> SVM
pipeline_svm = make_pipeline(TfidfVectorizer(), SVC(kernel='linear', C=1.0, probability=True))
pipeline_svm.fit(X_train_ml, y_train_ml)

# Verify Accuracy
y_pred_svm = pipeline_svm.predict(X_test_ml)
acc_svm = accuracy_score(y_test_ml, y_pred_svm)
print(f"Scheme 1 (SVM) Accuracy: {acc_svm:.4f}")
print(classification_report(y_test_ml, y_pred_svm, target_names=['Negative', 'Positive', 'Spam']))

Scheme 1 (SVM) Accuracy: 0.9955
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00       226
    Positive       1.00      0.99      0.99       225
        Spam       0.99      1.00      0.99       221

    accuracy                           1.00       672
   macro avg       1.00      1.00      1.00       672
weighted avg       1.00      1.00      1.00       672



### Scheme 2: Random Forest + CountVectorizer (80/20 Tuning)

In [5]:
# Pipeline: CountVectorizer -> Random Forest
pipeline_rf = make_pipeline(CountVectorizer(), RandomForestClassifier(n_estimators=100, random_state=42))
pipeline_rf.fit(X_train_ml, y_train_ml)

# Verify Accuracy
y_pred_rf = pipeline_rf.predict(X_test_ml)
acc_rf = accuracy_score(y_test_ml, y_pred_rf)
print(f"Scheme 2 (Random Forest) Accuracy: {acc_rf:.4f}")
print(classification_report(y_test_ml, y_pred_rf, target_names=['Negative', 'Positive', 'Spam']))

Scheme 2 (Random Forest) Accuracy: 0.9985
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00       226
    Positive       1.00      1.00      1.00       225
        Spam       1.00      1.00      1.00       221

    accuracy                           1.00       672
   macro avg       1.00      1.00      1.00       672
weighted avg       1.00      1.00      1.00       672



### Scheme 3: Deep Learning (Bi-LSTM)

In [6]:
VOCAB_SIZE = 5000
MAX_LENGTH = 100
EMBEDDING_DIM = 64

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df_clean['cleaned_text'])

sequences = tokenizer.texts_to_sequences(df_clean['cleaned_text'])
padded_sequences = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')

X_dl = padded_sequences
y_dl = df_clean['label'].values

# Split Data (70/30 for Deep Learning Variation)
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(X_dl, y_dl, test_size=0.3, random_state=42, stratify=y_dl)
print(f"DL Train shape: {X_train_dl.shape}, Test shape: {X_test_dl.shape}")

DL Train shape: (2352, 100), Test shape: (1008, 100)


In [7]:
model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.4),
    Bidirectional(LSTM(64)),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax') # 3 Output Classes
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Callbacks
checkpoint = ModelCheckpoint("best_model.keras", monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-6)

history = model.fit(
    X_train_dl, y_train_dl,
    epochs=20,
    batch_size=32,
    validation_data=(X_test_dl, y_test_dl),
    callbacks=[checkpoint, early_stop, reduce_lr]
)

/home/asephs/venv/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1770729707.309633  807586 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3184 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Epoch 1/20


2026-02-10 20:21:51.273864: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002



 1/74 ━━━━━━━━━━━━━━━━━━━━ 5:21 4s/step - accuracy: 0.3750 - loss: 1.0983


 5/74 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3461 - loss: 1.0981


 9/74 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3408 - loss: 1.0989

... (205 outputs left out)


69/74 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9997 - loss: 9.3476e-04


72/74 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9997 - loss: 9.4182e-04


Epoch 10: val_accuracy did not improve from 0.99206



74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9996 - loss: 0.0011 - val_accuracy: 0.9901 - val_loss: 0.0568 - learning_rate: 2.5000e-04


In [8]:
# Load best model
best_model = load_model("best_model.keras")
loss, acc_dl = best_model.evaluate(X_test_dl, y_test_dl)
print(f"Scheme 3 (Bi-LSTM) Accuracy: {acc_dl:.4f}")

y_pred_probs = best_model.predict(X_test_dl)
y_pred_dl = np.argmax(y_pred_probs, axis=1)
print(classification_report(y_test_dl, y_pred_dl, target_names=['Negative', 'Positive', 'Spam']))


 1/32 ━━━━━━━━━━━━━━━━━━━━ 15s 508ms/step - accuracy: 0.9688 - loss: 0.2995


 7/32 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9742 - loss: 0.2514  


14/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9812 - loss: 0.1720 


20/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9839 - loss: 0.1413


28/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9863 - loss: 0.1162


32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9921 - loss: 0.0470


... (4 outputs left out)


28/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step


              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00       339
    Positive       1.00      0.98      0.99       338
        Spam       0.98      1.00      0.99       331

    accuracy                           0.99      1008
   macro avg       0.99      0.99      0.99      1008
weighted avg       0.99      0.99      0.99      1008



## 4. Comparison of Results

In [9]:
results = pd.DataFrame({
    'Model': ['SVM + TF-IDF', 'RF + BOW', 'Bi-LSTM'],
    'Accuracy': [acc_svm, acc_rf, acc_dl],
    'Target > 85%': [acc_svm > 0.85, acc_rf > 0.85, acc_dl > 0.85],
    'DeepTarget > 92%': [False, False, acc_dl > 0.92]
})
print(results)

          Model  Accuracy  Target > 85%  DeepTarget > 92%
0  SVM + TF-IDF  0.995536          True             False
1      RF + BOW  0.998512          True             False
2       Bi-LSTM  0.992063          True              True


## 5. Inference / Testing
Test the model with custom sentences.

In [10]:
def predict_sentiment(text_list):
    # Preprocess
    cleaned_texts = [clean_text(t) for t in text_list]
    sequences = tokenizer.texts_to_sequences(cleaned_texts)
    padded = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')
    
    # Predict
    probs = best_model.predict(padded)
    preds = np.argmax(probs, axis=1)
    
    labels = {0: 'Negative', 1: 'Positive', 2: 'Spam'}
    results = []
    for t, p in zip(text_list, preds):
        results.append({'Text': t, 'Prediction': labels[p]})
    
    return pd.DataFrame(results)

# Test Cases
test_sentences = [
    "Video ini sangat bermanfaat, terima kasih bang!",
    "Konten sampah, tidak mendidik sama sekali.",
    "Cek ig kami kak, ada promo menarik.",
    "Biasa saja sih, tidak ada yang spesial.",
    "Gajelas lu bang, unsub ah."
]

inference_results = predict_sentiment(test_sentences)
print(inference_results)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step


                                              Text Prediction
0  Video ini sangat bermanfaat, terima kasih bang!       Spam
1       Konten sampah, tidak mendidik sama sekali.       Spam
2              Cek ig kami kak, ada promo menarik.       Spam
3          Biasa saja sih, tidak ada yang spesial.       Spam
4                       Gajelas lu bang, unsub ah.   Positive
